# Drug Discovery OpenEnv Training Notebook

This notebook is the end-to-end training and evaluation workflow for the live-only drug discovery environment.

It covers:
- install and runtime checks
- live API smoke tests
- inspecting the actual training samples
- baseline evaluation with step-by-step logs
- multi-disease GRPO training with debug traces
- evaluation of the trained checkpoint
- artifact inspection for README plots and submission evidence


In [ ]:
DISEASES = [
    "Type 2 Diabetes",
    "Non-small cell lung cancer",
]
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "outputs/grpo"
DEBUG_LIMIT = 6


## 1. Install dependencies

Use a T4 or better GPU. Internet access must be enabled because the environment is `live_only`.

In [ ]:
!python -m pip install -U pip
!pip install -e .[training,test]
!pip install -U "trl>=0.19.0" "transformers>=4.50.0" "accelerate>=1.0.0" datasets peft

In [ ]:
import os
os.environ["PYTHONUTF8"] = "1"

import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")

## 2. Verify the repo and live provider

In [ ]:
!pytest -q

In [ ]:
!python -u -m drug_discovery_env.scripts.live_smoke --disease "Type 2 Diabetes" --literature-query "INSR diabetes selectivity safety"

## 3. Inspect the actual offline training samples

This shows the prompt sent to the model, the bootstrap completion, the disease, and the reward attached to that step.

In [ ]:
from drug_discovery_env.training.rollout_generator import generate_rollouts

samples = generate_rollouts(num_episodes=2, diseases=DISEASES)
print("num samples:", len(samples))
for i, sample in enumerate(samples[:3], 1):
    print(f"\n--- sample {i} ---")
    print("disease:", sample.disease)
    print("prompt:\n", sample.prompt[:1500])
    print("\ncompletion:\n", sample.completion)
    print("\nreward:", sample.reward)
    print("history:", sample.history)


## 4. Baseline evaluation with full intermediate logs

In [ ]:
!python -u -m drug_discovery_env.scripts.run_evaluation --disease "Type 2 Diabetes" --episodes 1 --policy scripted --verbose --debug-io

In [ ]:
!python -u -m drug_discovery_env.scripts.run_evaluation --disease "Type 2 Diabetes" --episodes 1 --policy model --model Qwen/Qwen2.5-0.5B-Instruct --device cuda --verbose --debug-io

## 5. Multi-disease GRPO training with replay traces

The `--debug-io` flag prints what the trainer is scoring: prompt, completion, parsed action, env result, next state, and reward.

In [ ]:
disease_arg = ",".join(DISEASES)
print(disease_arg)

In [ ]:
import subprocess

subprocess.run([
    'python', '-u', '-m', 'drug_discovery_env.scripts.run_grpo',
    '--train',
    '--diseases', disease_arg,
    '--model', MODEL_NAME,
    '--episodes', '4',
    '--device', 'cuda',
    '--output-dir', OUTPUT_DIR,
    '--max-train-steps', '20',
    '--debug-io',
    '--debug-limit', str(DEBUG_LIMIT),
], check=True)

## 6. Evaluate the trained checkpoint

In [ ]:
subprocess.run([
    'python', '-u', '-m', 'drug_discovery_env.scripts.run_evaluation',
    '--disease', 'Type 2 Diabetes',
    '--episodes', '1',
    '--policy', 'model',
    '--model', OUTPUT_DIR,
    '--device', 'cuda',
    '--verbose',
    '--debug-io',
], check=True)

## 7. Build reproducible plots and summary artifacts

This produces the PNGs and JSON summary to reference in the README.

In [ ]:
subprocess.run([
    'python', '-u', '-m', 'drug_discovery_env.scripts.run_training_experiment',
    '--diseases', disease_arg,
    '--episodes', '4',
    '--eval-episodes', '1',
    '--device', 'cuda',
    '--model', MODEL_NAME,
    '--max-train-steps', '20',
    '--out-dir', 'artifacts/training',
], check=True)

In [ ]:
from IPython.display import Image, display

display(Image('artifacts/training/loss_curve.png'))
display(Image('artifacts/training/reward_curve.png'))
display(Image('artifacts/training/baseline_vs_trained.png'))

In [ ]:
import json
from pathlib import Path

summary = json.loads(Path('artifacts/training/training_summary.json').read_text())
summary